<a href="https://colab.research.google.com/github/gman332/axix-ai/blob/main/NACC_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import pandas as pd
import numpy as np
import re
from datetime import datetime

# =============================
# 1. FILE SETUP & CONFIG
# =============================
INPUT_FILE = "cancer.csv"
DATE_STAMP = datetime.now().strftime("%Y%m%d")
MISSING_CODES = [-4, 9, 88, 99, 995, 996, 997, 998]
TOP_4_CANCERS = ["has_breast", "has_prostate", "has_melanoma", "has_colon"]

print("Initializing NACC Cancer Pipeline v5.0 (2-File Output with Cases Log)...")

# =============================
# 2. LOAD & FILTER DATA
# =============================
df = pd.read_csv(INPUT_FILE)

# Restrict to Visit 1 to prevent longitudinal overlaps
if "NACCVNUM" in df.columns:
    df = df[df["NACCVNUM"] == 1].copy()

# Standardize missing values
df = df.replace(MISSING_CODES, np.nan)

# Ensure NACCAGE is numeric for stratification
if "NACCAGE" in df.columns:
    df["NACCAGE"] = pd.to_numeric(df["NACCAGE"], errors="coerce")

# =============================
# 3. BASE CANCER STATUS
# =============================
if "CANCER" in df.columns:
    base_cancer_status = df["CANCER"].value_counts(dropna=False).reset_index()
    base_cancer_status.columns = ["Raw_Cancer_Status", "Patient_Count"]
    # Add context for the professors
    status_mapping = {0: "No Cancer", 1: "Primary", 2: "Metastatic", 8: "Not Assessed"}
    base_cancer_status["Clinical_Meaning"] = base_cancer_status["Raw_Cancer_Status"].map(status_mapping)
else:
    base_cancer_status = pd.DataFrame({"Message": ["CANCER column not found"]})

# =============================
# 4. ALZHEIMER OUTCOME MAPPING
# =============================
df["NACCALZD"] = df["NACCALZD"].astype(str).str.strip()
df["has_alzheimers"] = (df["NACCALZD"] == "Alzheimer").astype(int)
df["has_dementia_any"] = df["NACCALZD"].isin(["Dementia", "Alzheimer"]).astype(int)
df["is_normal"] = (df["NACCALZD"] == "Normal").astype(int)

# =============================
# 5. CLEAN CANCER TEXT
# =============================
df["CANCSITE"] = df["CANCSITE"].astype(str).str.lower().str.strip()
df.loc[df["CANCSITE"] == "nan", "CANCSITE"] = np.nan

# =============================
# 6. REGEX BUILDER & DICTIONARY
# =============================
def build_pattern(whole_words=None, prefix_stems=None, phrases=None):
    parts = []
    if whole_words: parts.extend([rf"\b{w}\b" for w in whole_words])
    if prefix_stems: parts.extend([rf"\b{stem}" for stem in prefix_stems])
    if phrases: parts.extend(phrases)
    return "|".join(parts)

cancer_patterns = {
    "has_breast": build_pattern(whole_words=["breast","dcis","ductal","lobular"], prefix_stems=["mastect","lumpect"], phrases=["breat ca","brent ca"]),
    "has_prostate": r"\bpr\w{0,2}stat\w*\b|\bpsa\b|\bpnca\b",
    "has_ovarian": build_pattern(prefix_stems=["ovar"]),
    "has_uterine": build_pattern(whole_words=["uterus","gyn"], prefix_stems=["uterin","endometr"]),
    "has_cervical": build_pattern(prefix_stems=["cervic"]),
    "has_fallopian_tube": build_pattern(whole_words=["fallopian"]),
    "has_choriocarcinoma": build_pattern(whole_words=["choriocarcinoma"]),
    "has_testicular": build_pattern(whole_words=["testicular","testicle"]),
    "has_penile": build_pattern(whole_words=["penile"]),
    "has_melanoma": build_pattern(prefix_stems=["melan"]),
    "has_skin_non_melanoma": build_pattern(whole_words=["squamous","scc","bcc","basal","basil","bassal","dermal","face"], prefix_stems=["skin","mohs","forehead","mole","scalp","lips?"]),
    "has_colon": build_pattern(whole_words=["colon","rectal","rectum","cecum","colorectal","anal","anus","ileum","appendix","carcinoid","intestinal","cholangiocarcinoma","neuroendocrine"], phrases=["large intestine"], prefix_stems=["appendice"]),
    "has_gastric": build_pattern(whole_words=["gastric","stomach","esophagus","gist"], prefix_stems=["esophag"]),
    "has_pancreatic": build_pattern(prefix_stems=["pancrea"]),
    "has_bladder": build_pattern(whole_words=["bladder","bluddler","urethral"]),
    "has_kidney": build_pattern(prefix_stems=["kidney","renal","nephro","rencal"]),
    "has_lung": build_pattern(whole_words=["lung","nsclc"], phrases=["small cell","adenocarcinoma lung"]),
    "has_leukemia": build_pattern(whole_words=["leukemia","leukaemia","aml","cll","cml"], phrases=["acute myeloid","chronic myeloid","blood cancer"], prefix_stems=["myeloproliferative"]),
    "has_lymphoma": build_pattern(whole_words=["lymphoma","hodgkin","nhl","waldenstrom","mycosis"], phrases=["non hodg"]),
    "has_multiple_myeloma": build_pattern(whole_words=["myeloma","mylenoma"], phrases=["bone marrow"]),
    "has_brain": build_pattern(whole_words=["brain","glioma","meningioma","clivus","spinal","pituitary"], prefix_stems=["frontal"]),
    "has_thyroid": build_pattern(whole_words=["thyroid"]),
    "has_head_neck": build_pattern(whole_words=["oral","tongue","throat","vocal","parotid","salivary","jaw","nose"], prefix_stems=["laryn","pharyn"]),
    "has_sarcoma": build_pattern(whole_words=["sarcoma"]),
    "has_thymoma": build_pattern(whole_words=["thymoma"]),
    "has_pheochromocytoma": build_pattern(whole_words=["pheochromocytoma"]),
    "has_ocular": build_pattern(whole_words=["eye"], phrases=["tear duct"]),
    "has_parathyroid": build_pattern(whole_words=["parathyroid"]),
    "has_unknown_primary": build_pattern(phrases=["unknown primary"])
}

# =============================
# 7. APPLY CLASSIFICATION (Single Pass)
# =============================
for col, pattern in cancer_patterns.items():
    df[col] = df["CANCSITE"].str.contains(pattern, na=False, regex=True).astype(int)

# Clinical Override: Melanoma priority over non-melanoma skin
df.loc[df["has_melanoma"] == 1, "has_skin_non_melanoma"] = 0

subtype_cols = list(cancer_patterns.keys())
df["has_any_cancer"] = (df[subtype_cols].sum(axis=1) > 0).astype(int)

# Aggregate Female Hormonal Cancers
df["has_female_hormonal"] = df[["has_breast", "has_ovarian", "has_uterine"]].max(axis=1)

# =============================
# 8. GENERATE EXPANDED REPORTS
# =============================

# A. Category Counts
category_data = []
for col in subtype_cols:
    category_data.append({"Cancer_Category": col, "Case_Count": df[col].sum()})

category_data.append({"Cancer_Category": "SPECIAL: Female Hormonal (Combined)", "Case_Count": df["has_female_hormonal"].sum()})
if "NACCAGE" in df.columns:
    category_data.append({"Cancer_Category": "SPECIAL: Breast Cancer (< 50 yrs)", "Case_Count": df[(df["has_breast"]==1) & (df["NACCAGE"] < 50)].shape[0]})
    category_data.append({"Cancer_Category": "SPECIAL: Breast Cancer (>= 50 yrs)", "Case_Count": df[(df["has_breast"]==1) & (df["NACCAGE"] >= 50)].shape[0]})

category_counts = pd.DataFrame(category_data).sort_values(by="Case_Count", ascending=False)

# B. Expanded Transparency Log
transparency_rows = []

def clean_regex_term(term):
    return term.replace(r"\b", "").replace("\\", "").replace("w{0,2}", "*").replace("w*", "*").replace("?", "").strip()

def get_cohort_type(category):
    if category in ["has_prostate", "has_testicular", "has_penile"]: return "Male Only"
    if category in ["has_breast", "has_ovarian", "has_uterine", "has_cervical", "has_fallopian_tube", "has_choriocarcinoma"]: return "Female Only"
    return "Both"

for category, pattern in cancer_patterns.items():
    total_cases = df[category].sum()
    raw_terms = re.split(r'\|', pattern)
    term_counts = {raw_term: 0 for raw_term in raw_terms}

    for _, row in df[df[category] == 1].iterrows():
        text = row["CANCSITE"]
        for raw_term in raw_terms:
            if pd.notna(text) and re.search(raw_term, text):
                term_counts[raw_term] += 1
                break

    total_term_hits = sum(term_counts.values())
    formatted_terms = [f"{clean_regex_term(term)} ({count})" for term, count in term_counts.items() if count > 0]

    logic_note = "Overrides general skin cancer." if category == "has_melanoma" else "Ignored if melanoma is present." if category == "has_skin_non_melanoma" else "Standard single-pass."

    transparency_rows.append({
        "Cancer_Category": category,
        "Priority_Group": "⭐ Top 4" if category in TOP_4_CANCERS else "Standard",
        "Cohort_Type": get_cohort_type(category),
        "Total_Cases": total_cases,
        "Total_Term_Hits": total_term_hits,
        "Math_Check_Diff": total_term_hits - total_cases,
        "Logic_Rules": logic_note,
        "Readable_Search_Terms": " | ".join([clean_regex_term(rt) for rt in raw_terms]),
        "Matched_Term_Breakdown": " ;  ".join(formatted_terms),
        "Raw_Regex_Used": pattern
    })

transparency_df = pd.DataFrame(transparency_rows).sort_values(by="Total_Cases", ascending=False)

# C. Cross-Contamination Matrix
overlap_matrix = df[subtype_cols].T.dot(df[subtype_cols])
overlap_matrix.index.name = "Co-occurring Cancers"

# D. Garbage-In / Data Warnings
warnings_list = []
if "SEX" in df.columns:
    females_prostate = df[(df["has_prostate"] == 1) & (df["SEX"].astype(str).str.lower().str.contains("female"))]
    for _, row in females_prostate.iterrows():
        nacc_id = row["NACCID"] if "NACCID" in df.columns else "Unknown"
        warnings_list.append({"NACCID": nacc_id, "Issue": "Female tagged with Prostate Cancer", "CANCSITE": row["CANCSITE"], "SEX": row["SEX"]})

    males_female_cancers = df[
        ((df["has_ovarian"] == 1) | (df["has_uterine"] == 1) | (df["has_cervical"] == 1)) &
        (df["SEX"].astype(str).str.lower().str.contains("male", na=False)) &
        (~df["SEX"].astype(str).str.lower().str.contains("female", na=False))
    ]
    for _, row in males_female_cancers.iterrows():
        nacc_id = row["NACCID"] if "NACCID" in df.columns else "Unknown"
        warnings_list.append({"NACCID": nacc_id, "Issue": "Male tagged with Female-only Cancer", "CANCSITE": row["CANCSITE"], "SEX": row["SEX"]})

warnings_df = pd.DataFrame(warnings_list) if warnings_list else pd.DataFrame([{"Message": "No biological sex anomalies detected."}])

# E. Pipeline Methodology Summary
methodology_df = pd.DataFrame({
    "Pipeline Component": ["Visit Filtering", "Missing Values", "Text Search Architecture", "Melanoma Override", "Math Check Validation", "Hormonal Stratification"],
    "Action Applied": ["Filtered to NACCVNUM == 1", "Converted -4, 9, 88, 99 codes to Null", "Single-pass regex string matching", "Melanoma nullifies non-melanoma skin flags", "Cross-referenced term hits vs total cases", "Grouped Female Breast, Ovarian, Uterine"],
    "Scientific Rationale": ["Prevents duplicate tracking of the same patient across longitudinal visits.", "Prevents numerical codes from skewing algorithmic counts.", "Prevents double-counting a single text entry.", "Ensures highest-severity diagnosis takes precedence.", "Proves zero double-counting artifacts in the code.", "Requested by Dr. McElligott to isolate hormone-mediated cases."]
})

# F. EXACT RAW CASES PER CATEGORY (Requested by user / Dr. Abduli's Quality Check)
matched_cases_rows = []
for category in subtype_cols:
    cases = df.loc[df[category] == 1, "CANCSITE"].value_counts().reset_index()
    cases.columns = ["Matched_CANCSITE_Text", "Frequency"]
    for _, row in cases.iterrows():
        matched_cases_rows.append({
            "Cancer_Category": category,
            "Matched_CANCSITE_Text": row["Matched_CANCSITE_Text"],
            "Frequency": row["Frequency"]
        })
matched_cases_df = pd.DataFrame(matched_cases_rows)


# =============================
# 9. EXPORT EXACTLY 2 FILES
# =============================

# File 1: CSV for the pure data
data_filename = f"Cancer_With_Indicators_{DATE_STAMP}.csv"
df.to_csv(data_filename, index=False)
print(f"✅ File 1 Created: Main data exported to: {data_filename}")

# File 2: EXCEL for the beautifully formatted, expanded summary
report_file_name = f"Cancer_Comprehensive_Summary_{DATE_STAMP}.xlsx"

with pd.ExcelWriter(report_file_name, engine='openpyxl') as writer:
    methodology_df.to_excel(writer, sheet_name="0_Pipeline_Methodology", index=False)
    base_cancer_status.to_excel(writer, sheet_name="1_Base_Cancer_Status", index=False)
    category_counts.to_excel(writer, sheet_name="2_Category_Counts", index=False)
    transparency_df.to_excel(writer, sheet_name="3_Transparency_Log", index=False)
    overlap_matrix.to_excel(writer, sheet_name="4_Cross_Contamination")
    warnings_df.to_excel(writer, sheet_name="5_Data_Warnings", index=False)
    matched_cases_df.to_excel(writer, sheet_name="6_Matched_Cases", index=False)

    # Auto-adjust column widths for professional appearance
    for sheetname in writer.sheets:
        worksheet = writer.sheets[sheetname]
        for col in worksheet.columns:
            max_length = 0
            column = col[0].column_letter
            for cell in col:
                try:
                    if len(str(cell.value)) > max_length: max_length = len(str(cell.value))
                except: pass
            worksheet.column_dimensions[column].width = min((max_length + 2), 70)

print(f"✅ File 2 Created: Comprehensive Report exported to: {report_file_name}")

Initializing NACC Cancer Pipeline v5.0 (2-File Output with Cases Log)...
✅ File 1 Created: Main data exported to: Cancer_With_Indicators_20260305.csv
✅ File 2 Created: Comprehensive Report exported to: Cancer_Comprehensive_Summary_20260305.xlsx
